In [ ]:
from typing import Any, Dict, List

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from dotenv import load_dotenv

load_dotenv()


reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = [file.parse() for file in files]
chunks = chunk_documents(documents, size=2000, step=1000)

print(chunks[0])  # Verify that content and filename exist

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(chunks)


class SearchTools:
    def __init__(self, index: Index):
        self.index = index
        self.search_count = 0

    def search(self, query: str) -> List[Dict[str, Any]]:
        self.search_count += 1
        print(f"SEARCH CALL #{self.search_count}: {query}")

        return self.index.search(
            query=query,
            boost_dict={"content": 3.0},
            num_results=5,
        )


search_tools = SearchTools(index)

tools = Tools()
tools.add_tools(search_tools)

developer_prompt = """
You're a course teaching assistant. Answer the student's question using 
the search tool. Make multiple searches with different keywords before answering.
"""

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=developer_prompt,
    chat_interface=IPythonChatInterface(),
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

runner.run()

{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone

In [ ]:
search_tools.search_count